# Robust Classification Workflow for `tt̄Z` vs `WZ`

This notebook is a full replacement for the original exploratory workflow. It has two goals:

1. **Audit the original notebook methodology** for leakage, robustness, and evaluation issues.
2. **Provide a leakage-aware, reproducible pipeline** with detailed EDA, cross-validation, hyperparameter optimization, and a final untouched holdout evaluation.

> **Important design principle:** all model selection decisions in this notebook are based on the training split only. The test split is used exactly once at the end for the final comparison.


## 1. Methodology audit of the original `project.ipynb`

The original notebook contains several good ideas (baseline model, AUC-driven evaluation, tree-based feature importance, and attempts at cross-validation), but it also has a number of methodological weaknesses that can bias performance estimates.

### Main issues identified

1. **Global outlier filtering before the train/test split**
   - The original notebook computes z-scores on the full dataset and removes rows before the split.
   - That means distributional information from future test examples influences which samples remain in training.
   - In addition, aggressive global z-score filtering can remove physically meaningful tails rather than true measurement errors.

2. **Feature selection outside a leakage-safe CV loop**
   - Top features are selected using a decision tree trained after several modeling steps.
   - Those selected features are then reused in later models, including the neural network.
   - Because feature selection is not nested inside cross-validation, downstream estimates can be optimistic.

3. **Repeated reuse of the same test set for model development**
   - The notebook checks decision-tree performance on the test set, then uses that information to continue iterating.
   - It later compares multiple network variants against the same test set.
   - Once the test set influences choices, it is no longer a true unseen estimate.

4. **Hyperparameter tuning on a reduced subset chosen after earlier analysis**
   - The tree search is run on a 30% subset derived after inspecting earlier learning curves.
   - That subset choice is not validated with uncertainty estimates and introduces unnecessary instability.
   - If compute is a concern, it is better to tune on the full training split with fewer parameter combinations or fewer folds.

5. **Inconsistent validation strategy across models**
   - The decision tree uses stratified CV in some places.
   - The neural network mainly relies on a single validation split rather than cross-validation.
   - The final neural-network CV uses plain `KFold` instead of `StratifiedKFold`, which is weaker for classification.

6. **Evaluation not fully aligned with the stated objective**
   - The text states that AUC is the primary metric, which is reasonable here.
   - However, accuracy is still highlighted in places, and threshold-based confusion matrices are discussed without jointly considering class balance, precision-recall behavior, and calibration.

7. **No clean separation between descriptive EDA and modeling decisions**
   - The original notebook mixes plotting, filtering, feature selection, tuning, and evaluation in a way that makes the workflow hard to audit and reproduce.

### How this notebook fixes those issues

- Split the data once into **train** and **test** using stratification.
- Keep preprocessing inside **pipelines** whenever it can affect model fitting.
- Use **repeated stratified cross-validation** for baseline comparison on the training split.
- Use **nested CV** for tuned-model comparison to reduce optimism.
- Use the **test split once** at the very end.
- Report more than one metric: ROC AUC, PR AUC, balanced accuracy, F1, precision, and recall.
- Add explicit markdown interpretation after each major analytical block.


## 2. Imports and reproducibility setup


In [ ]:
from __future__ import annotations

from pathlib import Path
from pprint import pprint

import warnings
warnings.filterwarnings("ignore")

import h5py
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from IPython.display import Markdown, display
from numpy.lib.recfunctions import structured_to_unstructured

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    auc,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import (
    RandomizedSearchCV,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

RANDOM_STATE = 42
DATA_DIR = Path("data")


## 3. Load and assemble the dataset

The original notebook loads structured arrays from HDF5 files. We keep that approach, but wrap it in a compact and reproducible data-loading function.


In [ ]:
def load_events(path: Path) -> pd.DataFrame:
    with h5py.File(path, "r") as handle:
        events = handle["events"][:]
    columns = list(events.dtype.names)
    array = structured_to_unstructured(events)
    return pd.DataFrame(array, columns=columns)

signal = load_events(DATA_DIR / "output_signal.h5")
background = load_events(DATA_DIR / "output_bg.h5")

signal["label"] = 1
background["label"] = 0

data = pd.concat([signal, background], ignore_index=True)

print(f"Signal rows     : {len(signal):,}")
print(f"Background rows : {len(background):,}")
print(f"Combined rows   : {len(data):,}")
print(f"Feature columns : {data.shape[1] - 1}")

data.head()


## 4. Basic data-quality checks

This section deliberately avoids any target-aware transformation. We only inspect the raw table structure, class balance, missingness, duplicates, and constant columns.


In [ ]:
summary = pd.DataFrame(
    {
        "dtype": data.dtypes.astype(str),
        "missing": data.isna().sum(),
        "missing_pct": data.isna().mean().mul(100),
        "n_unique": data.nunique(dropna=False),
    }
)
summary


In [ ]:
class_balance = data["label"].value_counts().sort_index().rename(index={0: "background", 1: "signal"})
class_balance_df = pd.DataFrame(
    {
        "count": class_balance,
        "fraction": (class_balance / len(data)).round(4),
    }
)
class_balance_df


In [ ]:
feature_cols = [col for col in data.columns if col != "label"]
constant_features = [col for col in feature_cols if data[col].nunique(dropna=False) <= 1]
duplicate_rows = data.duplicated().sum()

print("Constant features:", constant_features)
print(f"Duplicate rows: {duplicate_rows:,}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=data, x="label", ax=axes[0])
axes[0].set_xticklabels(["background", "signal"])
axes[0].set_title("Class balance")
axes[0].set_xlabel("")
axes[0].set_ylabel("count")

(summary.loc[feature_cols, "missing_pct"].sort_values(ascending=False)
 .plot(kind="bar", ax=axes[1], color="steelblue"))
axes[1].set_title("Missing-value percentage by feature")
axes[1].set_ylabel("missing %")
axes[1].set_xlabel("")
plt.tight_layout()
plt.show()


In [ ]:
quality_notes = []

if constant_features:
    quality_notes.append(
        f"- Constant features detected and removed before modeling: {', '.join(constant_features)}."
    )
else:
    quality_notes.append("- No constant features were detected.")

if summary.loc[feature_cols, "missing"].sum() == 0:
    quality_notes.append("- No missing values were detected in the feature matrix.")
else:
    quality_notes.append("- Missing values are present and should be handled inside the modeling pipeline.")

if duplicate_rows == 0:
    quality_notes.append("- No exact duplicate rows were detected.")
else:
    quality_notes.append(f"- {duplicate_rows:,} duplicate rows were detected and should be investigated.")

majority_share = class_balance.max() / len(data)
quality_notes.append(
    f"- The majority-class share is {majority_share:.2%}, so accuracy alone would be an incomplete metric."
)

display(Markdown("### Data-quality interpretation\n" + "\n".join(quality_notes)))


## 5. Leakage-safe train/test split

From this point on, the **test set is locked away** and never used for preprocessing choices, model selection, or hyperparameter tuning.


In [ ]:
model_data = data.drop(columns=constant_features).copy()
feature_cols = [col for col in model_data.columns if col != "label"]

X = model_data[feature_cols]
y = model_data["label"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
print("Train class balance:")
print(y_train.value_counts(normalize=True).sort_index())
print("Test class balance:")
print(y_test.value_counts(normalize=True).sort_index())


## 6. Exploratory data analysis (EDA)

The EDA below is intentionally descriptive. It does **not** delete examples, clip values, or select features for the final models. That is important: plots are used to understand the physics-inspired observables, not to introduce leakage.


In [ ]:
train_summary = X_train.describe().T
train_summary["skew_proxy"] = ((train_summary["mean"] - train_summary["50%"].astype(float)) / train_summary["std"].replace(0, np.nan)).round(3)
train_summary.sort_values("std", ascending=False).head(10)


In [ ]:
correlation = X_train.corr(numeric_only=True)
plt.figure(figsize=(12, 10))
sns.heatmap(correlation, cmap="coolwarm", center=0)
plt.title("Feature correlation heatmap (training split only)")
plt.tight_layout()
plt.show()


In [ ]:
mi = pd.Series(mutual_info_classif(X_train, y_train, random_state=RANDOM_STATE), index=X_train.columns)
mi = mi.sort_values(ascending=False)
mi.to_frame("mutual_information").head(12)


In [ ]:
feature_auc = {}
for col in X_train.columns:
    raw_auc = roc_auc_score(y_train, X_train[col])
    feature_auc[col] = max(raw_auc, 1 - raw_auc)

feature_auc = pd.Series(feature_auc).sort_values(ascending=False)
eda_top_features = feature_auc.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
for ax, col in zip(axes, eda_top_features):
    sns.kdeplot(data=pd.concat([X_train, y_train], axis=1), x=col, hue="label", common_norm=False, ax=ax)
    ax.set_title(f"{col} | single-feature ROC AUC ≈ {feature_auc[col]:.3f}")
for ax in axes[len(eda_top_features):]:
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()
plot_df = pd.concat([X_train[eda_top_features], y_train.rename("label")], axis=1)
for ax, col in zip(axes, eda_top_features):
    sns.boxplot(data=plot_df, x="label", y=col, ax=ax)
    ax.set_xticklabels(["background", "signal"])
    ax.set_title(f"{col} by class")
for ax in axes[len(eda_top_features):]:
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
eda_lines = [
    f"- The strongest univariate separators on the training split are: {', '.join(eda_top_features)}.",
    "- Several observables are visibly skewed, but skewness alone is not a reason to drop events.",
    "- Correlated features are present, which makes regularized linear models and tree ensembles useful complementary baselines.",
    "- Because class balance is not extreme but not perfectly even either, ROC AUC and PR AUC are both informative.",
]
display(Markdown("### EDA interpretation\n" + "\n".join(eda_lines)))


## 7. Modeling strategy

We compare four model families:

- **Dummy baseline** to anchor expectations.
- **Logistic regression** as a simple, well-regularized linear baseline.
- **Decision tree / random forest family** for non-linear, interpretable, feature-interaction-heavy behavior.
- **MLPClassifier** as a lightweight neural-network baseline that is easier to reproduce than a custom TensorFlow stack.
- **HistGradientBoostingClassifier** as a strong tabular-data benchmark.

All preprocessing that can affect fitting is placed **inside pipelines**.


In [ ]:
scoring = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1",
    "precision": "precision",
    "recall": "recall",
}

baseline_models = {
    "dummy": DummyClassifier(strategy="prior"),
    "logistic": Pipeline(
        [
            ("scale", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)),
        ]
    ),
    "decision_tree": Pipeline(
        [("model", __import__("sklearn.tree").tree.DecisionTreeClassifier(random_state=RANDOM_STATE))]
    ),
    "random_forest": Pipeline(
        [
            (
                "model",
                RandomForestClassifier(
                    n_estimators=300,
                    random_state=RANDOM_STATE,
                    n_jobs=-1,
                ),
            )
        ]
    ),
    "hist_gb": Pipeline(
        [("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE))]
    ),
    "mlp": Pipeline(
        [
            ("scale", StandardScaler()),
            (
                "model",
                MLPClassifier(
                    hidden_layer_sizes=(128, 64),
                    alpha=1e-4,
                    learning_rate_init=1e-3,
                    max_iter=300,
                    early_stopping=True,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    ),
}

baseline_cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=2, random_state=RANDOM_STATE)


In [ ]:
baseline_rows = []
for name, estimator in baseline_models.items():
    cv_result = cross_validate(
        estimator,
        X_train,
        y_train,
        cv=baseline_cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )
    row = {"model": name}
    for metric in scoring:
        row[f"mean_{metric}"] = cv_result[f"test_{metric}"].mean()
        row[f"std_{metric}"] = cv_result[f"test_{metric}"].std()
    baseline_rows.append(row)

baseline_results = pd.DataFrame(baseline_rows).sort_values("mean_roc_auc", ascending=False)
baseline_results


In [ ]:
display(
    Markdown(
        "### Baseline comparison interpretation\n"
        f"- The best mean CV ROC AUC on the training split is **{baseline_results.iloc[0]['model']}**.\n"
        f"- The dummy model provides the no-skill reference and quantifies how misleading simple accuracy-based evaluation can be.\n"
        f"- The spread (`std_roc_auc`) is included because robustness matters almost as much as the mean score."
    )
)


## 8. Hyperparameter optimization with nested cross-validation

To avoid optimistic estimates, tuned-model performance is assessed with **nested CV**:

- **Inner loop**: `RandomizedSearchCV` chooses hyperparameters.
- **Outer loop**: a fresh validation fold estimates tuned-model performance.

This is more computationally expensive than a single search, but it is the correct way to compare tuned models when robustness matters.


In [ ]:
search_spaces = {
    "logistic": (
        Pipeline([
            ("scale", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, solver="liblinear", random_state=RANDOM_STATE)),
        ]),
        {
            "model__C": np.logspace(-3, 2, 20),
            "model__penalty": ["l1", "l2"],
            "model__class_weight": [None, "balanced"],
        },
    ),
    "decision_tree": (
        Pipeline([
            ("model", __import__("sklearn.tree").tree.DecisionTreeClassifier(random_state=RANDOM_STATE)),
        ]),
        {
            "model__criterion": ["gini", "entropy", "log_loss"],
            "model__max_depth": [3, 5, 7, 10, None],
            "model__min_samples_split": [2, 5, 10, 20, 50],
            "model__min_samples_leaf": [1, 2, 5, 10, 20],
            "model__class_weight": [None, "balanced"],
        },
    ),
    "random_forest": (
        Pipeline([
            ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1)),
        ]),
        {
            "model__n_estimators": [200, 400, 600],
            "model__max_depth": [None, 6, 10, 16],
            "model__min_samples_split": [2, 5, 10, 20],
            "model__min_samples_leaf": [1, 2, 5, 10],
            "model__max_features": ["sqrt", "log2", 0.5, None],
            "model__class_weight": [None, "balanced"],
        },
    ),
    "hist_gb": (
        Pipeline([
            ("model", HistGradientBoostingClassifier(random_state=RANDOM_STATE)),
        ]),
        {
            "model__learning_rate": np.linspace(0.02, 0.2, 10),
            "model__max_depth": [None, 3, 5, 7],
            "model__max_leaf_nodes": [15, 31, 63, 127],
            "model__min_samples_leaf": [20, 50, 100, 200],
            "model__l2_regularization": np.logspace(-6, 1, 12),
        },
    ),
    "mlp": (
        Pipeline([
            ("scale", StandardScaler()),
            (
                "model",
                MLPClassifier(
                    max_iter=300,
                    early_stopping=True,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]),
        {
            "model__hidden_layer_sizes": [(64,), (128,), (128, 64), (256, 128)],
            "model__alpha": np.logspace(-6, -2, 10),
            "model__learning_rate_init": np.logspace(-4, -2, 10),
            "model__batch_size": [128, 256, 512],
        },
    ),
}

inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=RANDOM_STATE)
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)


In [ ]:
from collections import defaultdict

nested_rows = []
search_objects = {}

for name, (estimator, params) in search_spaces.items():
    search = RandomizedSearchCV(
        estimator=estimator,
        param_distributions=params,
        n_iter=20,
        scoring="roc_auc",
        n_jobs=-1,
        cv=inner_cv,
        refit=True,
        random_state=RANDOM_STATE,
        return_train_score=False,
    )
    search_objects[name] = search

    nested = cross_validate(
        search,
        X_train,
        y_train,
        cv=outer_cv,
        scoring=scoring,
        n_jobs=1,
        return_estimator=True,
    )

    row = {"model": name}
    for metric in scoring:
        row[f"nested_mean_{metric}"] = nested[f"test_{metric}"].mean()
        row[f"nested_std_{metric}"] = nested[f"test_{metric}"].std()
    nested_rows.append(row)

nested_results = pd.DataFrame(nested_rows).sort_values("nested_mean_roc_auc", ascending=False)
nested_results


In [ ]:
display(
    Markdown(
        "### Nested-CV interpretation\n"
        f"- The most robust tuned-model estimate comes from **{nested_results.iloc[0]['model']}** based on nested ROC AUC.\n"
        f"- Nested CV is usually slightly harsher than a single CV search, which is expected and desirable.\n"
        f"- The standard deviation across outer folds should be read as a stability indicator rather than noise to ignore."
    )
)


## 9. Final hyperparameter search on the full training split

After comparing tuned model families with nested CV, we refit one search per model on the **entire training split only**. This produces a final candidate for the untouched holdout test set.


In [ ]:
final_search_results = {}
best_params = {}

for name, search in search_objects.items():
    search.fit(X_train, y_train)
    final_search_results[name] = search
    best_params[name] = search.best_params_

best_params_df = pd.DataFrame(
    {name: pd.Series(params) for name, params in best_params.items()}
).T
best_params_df


In [ ]:
holdout_rows = []
holdout_predictions = {}

for name, search in final_search_results.items():
    estimator = search.best_estimator_
    proba = estimator.predict_proba(X_test)[:, 1] if hasattr(estimator, "predict_proba") else estimator.decision_function(X_test)
    preds = estimator.predict(X_test)

    holdout_predictions[name] = {"proba": proba, "pred": preds, "estimator": estimator}

    holdout_rows.append(
        {
            "model": name,
            "test_roc_auc": roc_auc_score(y_test, proba),
            "test_pr_auc": average_precision_score(y_test, proba),
            "test_balanced_accuracy": balanced_accuracy_score(y_test, preds),
            "test_f1": f1_score(y_test, preds),
            "test_precision": precision_score(y_test, preds),
            "test_recall": recall_score(y_test, preds),
        }
    )

holdout_results = pd.DataFrame(holdout_rows).sort_values("test_roc_auc", ascending=False)
holdout_results


In [ ]:
display(
    Markdown(
        "### Holdout-test interpretation\n"
        f"- The final test-set ROC AUC leader is **{holdout_results.iloc[0]['model']}**.\n"
        f"- Compare the holdout ranking to the nested-CV ranking above; strong agreement increases confidence that model selection was stable.\n"
        f"- If a model jumps dramatically on the holdout split relative to nested CV, that may indicate variance rather than a truly superior method."
    )
)


## 10. ROC, PR, and confusion-matrix diagnostics


In [ ]:
top_models = holdout_results["model"].head(3).tolist()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name in top_models:
    proba = holdout_predictions[name]["proba"]
    fpr, tpr, _ = roc_curve(y_test, proba)
    precision, recall, _ = precision_recall_curve(y_test, proba)
    axes[0].plot(fpr, tpr, label=f"{name} (AUC={roc_auc_score(y_test, proba):.3f})")
    axes[1].plot(recall, precision, label=f"{name} (AP={average_precision_score(y_test, proba):.3f})")

axes[0].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[0].set_title("ROC curves on holdout test set")
axes[0].set_xlabel("False positive rate")
axes[0].set_ylabel("True positive rate")
axes[0].legend()

baseline_precision = y_test.mean()
axes[1].hlines(baseline_precision, 0, 1, linestyle="--", color="gray", label="class prevalence")
axes[1].set_title("Precision-recall curves on holdout test set")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
for name in top_models:
    fig, ax = plt.subplots(figsize=(5, 4))
    cm = confusion_matrix(y_test, holdout_predictions[name]["pred"])
    ConfusionMatrixDisplay(cm, display_labels=["background", "signal"]).plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Confusion matrix: {name}")
    plt.tight_layout()
    plt.show()


In [ ]:
best_model_name = holdout_results.iloc[0]["model"]
best_estimator = holdout_predictions[best_model_name]["estimator"]

if hasattr(best_estimator, "named_steps") and "model" in best_estimator.named_steps:
    raw_model = best_estimator.named_steps["model"]
else:
    raw_model = best_estimator

if hasattr(raw_model, "feature_importances_"):
    importance = pd.Series(raw_model.feature_importances_, index=X_train.columns).sort_values(ascending=False).head(15)
    importance.plot(kind="barh")
    plt.title(f"Feature importance for best model: {best_model_name}")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
elif best_model_name in {"logistic"}:
    coef = pd.Series(np.abs(raw_model.coef_[0]), index=X_train.columns).sort_values(ascending=False).head(15)
    coef.plot(kind="barh")
    plt.title(f"Absolute coefficient magnitude for best model: {best_model_name}")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    perm = permutation_importance(best_estimator, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
    importance = pd.Series(perm.importances_mean, index=X_test.columns).sort_values(ascending=False).head(15)
    importance.plot(kind="barh")
    plt.title(f"Permutation importance for best model: {best_model_name}")
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()


## 11. Final discussion


In [ ]:
best_holdout = holdout_results.iloc[0]
best_nested = nested_results.iloc[0]

final_discussion = f"""
### Final takeaways

- The original notebook likely **overstated certainty** because preprocessing, feature selection, and model iteration were not fully isolated from later evaluation.
- In this revised workflow, the leading model on the holdout split is **{best_holdout['model']}** with ROC AUC **{best_holdout['test_roc_auc']:.4f}** and PR AUC **{best_holdout['test_pr_auc']:.4f}**.
- The best nested-CV model is **{best_nested['model']}** with mean nested ROC AUC **{best_nested['nested_mean_roc_auc']:.4f} ± {best_nested['nested_std_roc_auc']:.4f}**.
- Agreement between nested CV and the final holdout ranking is a strong sign that the implementation is reasonably robust.
- If the ranking differs, the likely explanation is **model variance** rather than a dramatic change in true generalization performance.

### Practical interpretation

- If interpretability is the priority, tree-based models and logistic regression remain useful because they expose feature effects more directly.
- If raw predictive performance is the priority, boosted trees and carefully tuned neural networks / MLPs are often the strongest tabular baselines.
- In a physics context, the next methodological improvements would usually be:
  1. uncertainty-aware evaluation,
  2. event weighting if relevant to the analysis design,
  3. calibration checks,
  4. and, if available, systematic-variation robustness studies.

### Recommendation

Use this notebook as the new reference workflow because it separates:
- descriptive analysis,
- training-only model development,
- leakage-safe hyperparameter tuning,
- and final holdout evaluation.
"""

display(Markdown(final_discussion))


## 12. Optional export helpers

If you want a lightweight artifact for reports, the cell below saves the main comparison tables.


In [ ]:
OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

baseline_results.to_csv(OUTPUT_DIR / "baseline_cv_results.csv", index=False)
nested_results.to_csv(OUTPUT_DIR / "nested_cv_results.csv", index=False)
holdout_results.to_csv(OUTPUT_DIR / "holdout_test_results.csv", index=False)
best_params_df.to_csv(OUTPUT_DIR / "best_params.csv")

print(f"Saved tabular outputs to: {OUTPUT_DIR.resolve()}")
